# Research Study: End-to-End Voice Deepfake Detection with RawNet2-Mini and Raw Waveforms
## ASVspoof 2019 Logical Access (LA) Benchmark

### Abstract and Scientific Background
Standard audio representations (e.g. Mel filterbanks, LFCC, STFT) rely on magnitude spectrograms and discard the phase component of speech signals. However, neural acoustic models and vocoders (WaveNet, WORLD, Griffin-Lim, Neural Source-Filter) inherently produce temporal irregularities, micro-jitter in fundamental pitch periods, and phase dispersion across phonetic boundaries.

To capture these subtle time-domain and phase-dependent synthetic cues, this study formulates **RawNet2-Mini**, an end-to-end deep architecture that operates directly on the raw 1D audio waveform ($[1, 64000]$ samples at 16,000 Hz, equivalent to 4.0 seconds of speech). The network incorporates:
1. **Parameterized Sinc-Convolutions (SincNet)**: Instead of arbitrary convolution kernels, the first layer synthesizes bandpass sinc filters parameterized by learnable lower and upper cutoff frequencies:
$$g(t, f_1, f_2) = 2f_2 \text{sinc}(2\pi f_2 t) - 2f_1 \text{sinc}(2\pi f_1 t)$$
2. **Instantaneous Energy Detection**: Absolute-value non-linear activation $|x|$ acts as an analytical envelope detector.
3. **Residual Raw Blocks with Feature Map Scaling (FMS)**: 1D residual blocks equipped with channel-wise self-gating to dynamically re-scale filter activations.
4. **Adaptive Dual Pooling & Latent Space Projection**: Global Average and Max Pooling coupled with a 64-dimensional latent embedding layer.

This notebook executes a complete end-to-end research workflow:
1. Hardware diagnostics and seed initialization
2. Fast multi-path dataset discovery (<0.02s) with flac directory pruning
3. Protocol parsing and class distribution audit
4. Pre-cleaning Exploratory Data Analysis (glottal wave zoom, power spectral density, phase jitter)
5. Waveform preprocessing and RawBoost time-domain augmentation pipeline
6. SincNet parameterized filterbank formulation and initial frequency response analysis
7. PyTorch Dataset with dynamic augmentation and WeightedRandomSampler
8. RawNet2-Mini architecture implementation with Feature Map Scaling
9. Focal Loss optimization with label smoothing and Cosine Annealing scheduler
10. Full 30-epoch training and validation tracking with real-time Dev EER
11. 17 standalone publication-grade diagnostic figures (300 DPI in `/kaggle/working/figures/`)
12. Comprehensive biometric evaluations (EER, minDCF, ROC, DET, PR, Confusion Matrix)
13. Granular attack vulnerability analysis (A01 through A06)
14. Post-training learned Sinc filter frequency response adaptation
15. 2D t-SNE latent representation clustering
16. 1D Temporal Saliency attribution and live single-file inference demo

## Step 1: Library Installation and Hardware Diagnostics

In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])

import os, glob, time, json, math, random
import numpy as np, pandas as pd
import soundfile as sf, librosa, scipy.fftpack as fft_
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    confusion_matrix, accuracy_score, precision_recall_fscore_support
)
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Compute Device: {device}")
if use_amp:
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

fig_dir = "/kaggle/working/figures"
os.makedirs(fig_dir, exist_ok=True)
print(f"Publication Figures Directory: {fig_dir}")


## Step 2: Dataset Discovery and Path Resolution

In [ ]:
def resolve_dataset():
    search_dirs = ["/kaggle/input", "/kaggle/working", "."]
    detected_inputs = []
    for s_dir in search_dirs:
        if os.path.exists(s_dir):
            try:
                items = os.listdir(s_dir)
                detected_inputs.append((s_dir, items))
            except Exception:
                pass

    print("Detected Input Directories:")
    for s_dir, items in detected_inputs:
        print(f"  {s_dir}: {items}")

    found_protos = {"train": None, "dev": None, "eval": None}
    found_flacs = {"train": None, "dev": None, "eval": None}

    candidate_roots = [
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset",
        "/kaggle/input/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/asvpoof-2019-dataset/LA",
        "/kaggle/input/asvpoof-2019-dataset",
        "/kaggle/input/asvspoof-2019-dataset/LA/LA",
        "/kaggle/input/asvspoof-2019-dataset/LA",
        "/kaggle/input/asvspoof-2019-dataset",
        "/kaggle/input/asvspoof-2019/LA/LA",
        "/kaggle/input/asvspoof-2019/LA",
        "/kaggle/input/asvspoof-2019",
        "/kaggle/input/asvspoof2019/LA/LA",
        "/kaggle/input/asvspoof2019/LA",
        "/kaggle/input/asvspoof2019"
    ]

    for cand in candidate_roots:
        if os.path.isdir(cand):
            for part, sfx in [("train", "trn"), ("dev", "trl"), ("eval", "trl")]:
                direct_proto = os.path.join(cand, "ASVspoof2019_LA_cm_protocols", f"ASVspoof2019.LA.cm.{part}.{sfx}.txt")
                if os.path.isfile(direct_proto) and not found_protos[part]:
                    found_protos[part] = direct_proto
                direct_flac = os.path.join(cand, f"ASVspoof2019_LA_{part}", "flac")
                if os.path.isdir(direct_flac) and not found_flacs[part]:
                    found_flacs[part] = direct_flac
                elif os.path.isdir(os.path.join(cand, f"ASVspoof2019_LA_{part}")) and not found_flacs[part]:
                    found_flacs[part] = os.path.join(cand, f"ASVspoof2019_LA_{part}")

    needs_walk = any(v is None for v in [found_protos["train"], found_protos["dev"], found_flacs["train"], found_flacs["dev"]])
    if needs_walk:
        for s_dir in search_dirs:
            if not os.path.exists(s_dir):
                continue
            for root, dirs, files in os.walk(s_dir, followlinks=True):
                for f in files:
                    fl = f.lower()
                    if "cm" in fl and fl.endswith(".txt") and not f.startswith("._"):
                        if "train" in fl and ("trn" in fl or "train" in fl):
                            if not found_protos["train"]:
                                found_protos["train"] = os.path.join(root, f)
                        elif "dev" in fl and ("trl" in fl or "dev" in fl):
                            if not found_protos["dev"]:
                                found_protos["dev"] = os.path.join(root, f)
                        elif "eval" in fl and ("trl" in fl or "eval" in fl):
                            if not found_protos["eval"]:
                                found_protos["eval"] = os.path.join(root, f)

                for d in list(dirs):
                    dl = d.lower()
                    for part in ["train", "dev", "eval"]:
                        if (f"la_{part}" in dl or f"la.{part}" in dl or f"_{part}" in dl) and not found_flacs[part]:
                            sub_flac = os.path.join(root, d, "flac")
                            if os.path.isdir(sub_flac):
                                found_flacs[part] = sub_flac
                            elif os.path.isdir(os.path.join(root, d)):
                                found_flacs[part] = os.path.join(root, d)

                if "flac" in dirs:
                    dirs.remove("flac")

    return found_protos, found_flacs

proto_files, flac_dirs = resolve_dataset()

print("Resolved Dataset Resources:")
for k in ["train", "dev", "eval"]:
    p_path = proto_files[k]
    f_path = flac_dirs[k]
    p_ok = os.path.isfile(p_path) if p_path else False
    f_ok = os.path.isdir(f_path) if f_path else False
    f_count = len(os.listdir(f_path)) if f_ok else 0
    print(f"  [{k.upper()}] Protocol: {'OK' if p_ok else 'MISSING'} ({p_path})")
    print(f"          Audio:    {'OK' if f_ok else 'MISSING'} ({f_count:,} files in {f_path})")

if not proto_files["train"] or not os.path.isfile(proto_files["train"]):
    print("CRITICAL: ASVspoof 2019 dataset not detected in /kaggle/input.")
    print("Action required in Kaggle:")
    print("1. Click '+ Add Input' in the right sidebar panel.")
    print("2. Search for 'asvpoof-2019-dataset' (by awsaf49).")
    print("3. Click '+' to attach the dataset to this notebook.")
    print("4. Re-run this cell once the dataset is attached.")
    raise FileNotFoundError("Missing ASVspoof 2019 train protocol file. Please attach the dataset to Kaggle notebook.")

if not flac_dirs["train"] or not os.path.isdir(flac_dirs["train"]):
    raise FileNotFoundError("Missing ASVspoof 2019 train audio directory. Please verify dataset attachment.")


## Step 3: Protocol Parsing and Metadata Manifest Construction

In [ ]:
rows = []
for partition, path in proto_files.items():
    if not path or not os.path.exists(path):
        continue
    flac_folder = flac_dirs.get(partition)
    if not flac_folder or not os.path.exists(flac_folder):
        continue
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            spk, aid, env, atk, key = parts[0], parts[1], parts[2], parts[3], parts[4]
            rows.append({
                "speaker_id": spk,
                "audio_id": aid,
                "environment_id": env,
                "attack_id": atk,
                "key": key,
                "is_spoof": 1 if key == "spoof" else 0,
                "partition": partition,
                "file_path": os.path.join(flac_folder, f"{aid}.flac")
            })

manifest = pd.DataFrame(rows)
assert len(manifest) > 0, "Protocol parsing produced 0 records. Check dataset paths."
print(f"Total Parsed Records: {len(manifest):,}")

summary_table = []
for part in ["train", "dev", "eval"]:
    sub = manifest[manifest["partition"] == part]
    if len(sub) == 0:
        continue
    bon = int((sub["key"] == "bonafide").sum())
    spf = int((sub["key"] == "spoof").sum())
    summary_table.append({
        "Partition": part,
        "Total Utterances": len(sub),
        "Bonafide": bon,
        "Spoof": spf,
        "Spoof:Bonafide Ratio": f"{spf / max(bon, 1):.2f}:1"
    })
print(pd.DataFrame(summary_table).to_string(index=False))


## Step 4: Exploratory Data Analysis - Class Distribution Breakdown

In [ ]:
plt.figure(figsize=(10, 5))
partitions = ["train", "dev", "eval"]
bon_counts = [int(((manifest["partition"] == p) & (manifest["key"] == "bonafide")).sum()) for p in partitions]
spf_counts = [int(((manifest["partition"] == p) & (manifest["key"] == "spoof")).sum()) for p in partitions]

x_pos = np.arange(len(partitions))
width = 0.35

plt.bar(x_pos - width / 2, bon_counts, width, label="Bonafide (Authentic)", color="steelblue")
plt.bar(x_pos + width / 2, spf_counts, width, label="Spoof (Synthetic)", color="firebrick")

plt.xlabel("Dataset Partition", fontsize=12)
plt.ylabel("Number of Utterances", fontsize=12)
plt.title("ASVspoof 2019 LA Class Distribution Across Partitions", fontsize=13)
plt.xticks(x_pos, [p.upper() for p in partitions], fontsize=11)
plt.ylim(0, max(spf_counts) * 1.12)
plt.legend(fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.4)

offset = max(spf_counts) * 0.015
for i in range(len(partitions)):
    plt.text(x_pos[i] - width / 2, bon_counts[i] + offset, f"{bon_counts[i]:,}", ha="center", fontsize=9)
    plt.text(x_pos[i] + width / 2, spf_counts[i] + offset, f"{spf_counts[i]:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "01_class_distribution_breakdown.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 5: Exploratory Data Analysis - Speaker Demographics Profile

In [ ]:
train_spks = set(manifest[manifest["partition"] == "train"]["speaker_id"].unique())
dev_spks = set(manifest[manifest["partition"] == "dev"]["speaker_id"].unique())
eval_spks = set(manifest[manifest["partition"] == "eval"]["speaker_id"].unique())

print(f"Unique Speakers - Train: {len(train_spks)} | Dev: {len(dev_spks)} | Eval: {len(eval_spks)}")
overlap_train_dev = train_spks.intersection(dev_spks)
overlap_train_eval = train_spks.intersection(eval_spks)
overlap_dev_eval = dev_spks.intersection(eval_spks)

print(f"Speaker Overlap (Train & Dev):  {len(overlap_train_dev)} (Expected: 0)")
print(f"Speaker Overlap (Train & Eval): {len(overlap_train_eval)} (Expected: 0)")
print(f"Speaker Overlap (Dev & Eval):   {len(overlap_dev_eval)} (Expected: 0)")
assert len(overlap_train_dev) == 0 and len(overlap_train_eval) == 0

dev_sub = manifest[manifest["partition"] == "dev"]
spk_counts = dev_sub["speaker_id"].value_counts()

plt.figure(figsize=(9, 4.5))
plt.bar(spk_counts.index, spk_counts.values, color="teal", width=0.6)
plt.xlabel("Speaker Alphanumeric ID (Development Partition)", fontsize=11)
plt.ylabel("Total Utterances Count", fontsize=11)
plt.title("Utterances per Speaker in ASVspoof 2019 Development Partition", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "02_speaker_utterance_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 6: Exploratory Data Analysis - Time-Domain Micro-Structure and Pitch Period Jitter

In [ ]:
bon_candidates = manifest[(manifest["partition"] == "train") & (manifest["key"] == "bonafide")]
spf_candidates = manifest[(manifest["partition"] == "train") & (manifest["key"] == "spoof")]

sample_bon = None
for _, row in bon_candidates.iterrows():
    if os.path.isfile(row["file_path"]):
        sample_bon = row
        break
if sample_bon is None:
    sample_bon = bon_candidates.iloc[0]

sample_spf = None
for _, row in spf_candidates.iterrows():
    if os.path.isfile(row["file_path"]):
        sample_spf = row
        break
if sample_spf is None:
    sample_spf = spf_candidates.iloc[0]

def read_mono_audio(path, target_sr=16000):
    y, sr = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    return y

y_bon = read_mono_audio(sample_bon["file_path"])
y_spf = read_mono_audio(sample_spf["file_path"])

start_sample = int(1.0 * 16000)
zoom_len = int(0.04 * 16000)
t_zoom = np.linspace(0, 40, zoom_len)

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)

axes[0].plot(t_zoom, y_bon[start_sample:start_sample + zoom_len], color="steelblue", lw=1.2)
axes[0].set_title(f"Authentic Speech Micro-Waveform: {sample_bon['audio_id']} (Smooth Vocal Tract Glottal Pulses)", fontsize=11)
axes[0].set_ylabel("Amplitude", fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_zoom, y_spf[start_sample:start_sample + zoom_len], color="firebrick", lw=1.2)
axes[1].set_title(f"Synthetic Speech Micro-Waveform: {sample_spf['audio_id']} (Attack {sample_spf['attack_id']}: Vocoder Phase Discontinuity)", fontsize=11)
axes[1].set_xlabel("Time (milliseconds)", fontsize=10)
axes[1].set_ylabel("Amplitude", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "03_raw_waveform_glottal_anatomy.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 7: Exploratory Data Analysis - Power Spectral Density and Energy Distribution

In [ ]:
fft_size = 2048
freq_axis = np.linspace(0, 8000, fft_size // 2 + 1)

def compute_psd(sig):
    windowed = sig[:fft_size] * np.hanning(fft_size)
    spectrum = np.abs(np.fft.rfft(windowed)) ** 2
    return 10 * np.log10(spectrum + 1e-9)

psd_bon = compute_psd(y_bon[start_sample:])
psd_spf = compute_psd(y_spf[start_sample:])

plt.figure(figsize=(11, 4.5))
plt.plot(freq_axis, psd_bon, color="steelblue", lw=1.5, label=f"Bonafide ({sample_bon['audio_id']})")
plt.plot(freq_axis, psd_spf, color="firebrick", lw=1.5, linestyle="--", label=f"Spoof {sample_spf['attack_id']} ({sample_spf['audio_id']})")

plt.xlabel("Acoustic Frequency (Hz)", fontsize=11)
plt.ylabel("Power Spectral Density (dB/Hz)", fontsize=11)
plt.title("Power Spectral Density (PSD) Comparison: Authentic vs Synthetic Audio", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "04_power_spectral_density_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 8: Audio Preprocessing Pipeline - VAD, Pre-Emphasis and 64,000 Sample Windowing

In [ ]:
def preprocess_raw_audio(y, is_train=False, target_len=64000, alpha=0.97, top_db=40):
    y = np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])
    intervals = librosa.effects.split(y=y, top_db=top_db)
    if len(intervals):
        trimmed = np.concatenate([y[s:e] for s, e in intervals])
        if len(trimmed) > 1000:
            y = trimmed
    n_samples = len(y)
    if n_samples >= target_len:
        start = np.random.randint(0, n_samples - target_len + 1) if is_train else (n_samples - target_len) // 2
        y = y[start:start + target_len]
    else:
        y = np.pad(y, (0, target_len - n_samples), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

raw_sample = read_mono_audio(sample_bon["file_path"])
proc_sample = preprocess_raw_audio(raw_sample, is_train=False)
print(f"Raw Input Samples:        {len(raw_sample):,}")
print(f"Preprocessed Waveform:    {len(proc_sample):,} (Expected: 64,000 = 4.0s @ 16kHz)")
assert len(proc_sample) == 64000


## Step 9: RawBoost Waveform Augmentation Pipeline

In [ ]:
def augment_waveform(y, p_noise=0.5, p_mask=0.5):
    y_aug = y.copy()
    if np.random.rand() < p_noise:
        noise = np.random.randn(len(y_aug)).astype(np.float32)
        rms_y = np.sqrt(np.mean(y_aug ** 2) + 1e-8)
        rms_n = np.sqrt(np.mean(noise ** 2) + 1e-8)
        snr_db = np.random.uniform(15, 30)
        scale = (rms_y / rms_n) * (10 ** (-snr_db / 20))
        y_aug = y_aug + scale * noise
    if np.random.rand() < p_mask:
        mask_len = np.random.randint(800, 3200)
        start = np.random.randint(0, len(y_aug) - mask_len)
        y_aug[start:start + mask_len] = 0.0
    return y_aug

demo_aug = augment_waveform(proc_sample, p_noise=1.0, p_mask=1.0)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(proc_sample[:8000], color="steelblue", lw=0.9)
axes[0].set_title("Preprocessed Clean Waveform (First 0.5s = 8,000 samples)", fontsize=11)
axes[0].set_ylabel("Amplitude", fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(demo_aug[:8000], color="forestgreen", lw=0.9)
axes[1].set_title("RawBoost Augmented Waveform (Additive SNR Noise and Temporal Zero-Masking)", fontsize=11)
axes[1].set_xlabel("Audio Sample Index", fontsize=10)
axes[1].set_ylabel("Amplitude", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "05_rawboost_augmentation_stages.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 10: Parameterized Sinc-Convolutional Frontend (SincConv1D) Formulation

In [ ]:
class SincConv1D(nn.Module):
    def __init__(self, out_channels=64, kernel_size=129, sample_rate=16000, min_low_hz=50, min_band_hz=50):
        super().__init__()
        self.out_channels = out_channels
        self.kernel_size = kernel_size if kernel_size % 2 != 0 else kernel_size + 1
        self.sample_rate = sample_rate
        self.min_low_hz = min_low_hz
        self.min_band_hz = min_band_hz
        low_hz = 30
        high_hz = self.sample_rate / 2 - (self.min_low_hz + self.min_band_hz)
        mel = np.linspace(2595 * np.log10(1 + low_hz / 700), 2595 * np.log10(1 + high_hz / 700), self.out_channels + 1)
        hz = 700 * (10 ** (mel / 2595) - 1)
        self.low_hz_ = nn.Parameter(torch.Tensor(hz[:-1]).view(-1, 1))
        self.band_hz_ = nn.Parameter(torch.Tensor(np.diff(hz)).view(-1, 1))
        n_lin = torch.linspace(0, (self.kernel_size / 2) - 1, steps=int((self.kernel_size / 2)))
        window = 0.54 - 0.46 * torch.cos(2 * np.pi * n_lin / self.kernel_size)
        self.register_buffer("window_", window)
        n = (self.kernel_size - 1) / 2.0
        n_ = 2 * np.pi * torch.arange(-n, 0).view(1, -1) / self.sample_rate
        self.register_buffer("n_", n_)

    def get_filters(self):
        low = self.min_low_hz + torch.abs(self.low_hz_)
        high = torch.clamp(low + self.min_band_hz + torch.abs(self.band_hz_), self.min_low_hz, self.sample_rate / 2)
        band = (high - low)[:, 0]
        f_times_t_low = torch.matmul(low, self.n_)
        f_times_t_high = torch.matmul(high, self.n_)
        band_pass_left = ((torch.sin(f_times_t_high) - torch.sin(f_times_t_low)) / (self.n_ / 2)) * self.window_
        band_pass_center = 2 * band.view(-1, 1)
        band_pass_right = torch.flip(band_pass_left, dims=[1])
        band_pass = torch.cat([band_pass_left, band_pass_center, band_pass_right], dim=1)
        band_pass = band_pass / (2 * band[:, None])
        return band_pass.view(self.out_channels, 1, self.kernel_size)

    def forward(self, waveforms):
        filters = self.get_filters()
        return nn.functional.conv1d(waveforms, filters, stride=1, padding=self.kernel_size // 2)

sinc_demo = SincConv1D(out_channels=64, kernel_size=129)
test_wave = torch.randn(2, 1, 64000)
sinc_out = sinc_demo(test_wave)
print(f"SincConv1D Input Shape:  {test_wave.shape}")
print(f"SincConv1D Output Shape: {sinc_out.shape} (Expected: (2, 64, 64000))")
assert sinc_out.shape == (2, 64, 64000)


## Step 11: Initialized Sinc Filterbank Frequency Response Curves

In [ ]:
init_filters = sinc_demo.get_filters().squeeze().detach().cpu().numpy()
H_init = np.abs(np.fft.rfft(init_filters, n=1024, axis=-1))
freq_axis_sinc = np.linspace(0, 8000, H_init.shape[-1])

plt.figure(figsize=(11, 4.5))
for i in range(0, 64, 3):
    plt.plot(freq_axis_sinc, H_init[i], lw=1.2, alpha=0.75)

plt.xlabel("Frequency (Hz)", fontsize=11)
plt.ylabel("Magnitude Response", fontsize=11)
plt.title("Initial SincNet Parameterized Bandpass Filterbanks (Mel-Scale Initialization)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "06_sincnet_initial_frequency_responses.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 12: PyTorch Dataset Formulation with Dynamic Waveform Augmentation

In [ ]:
class RawDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            raw = read_mono_audio(row["file_path"])
            proc = preprocess_raw_audio(raw, is_train=self.is_train)
        except Exception:
            proc = np.zeros(64000, dtype=np.float32)

        if self.is_train:
            proc = augment_waveform(proc, p_noise=0.5, p_mask=0.5)

        tensor_x = torch.from_numpy(proc).unsqueeze(0)
        tensor_y = torch.tensor(int(row["is_spoof"]), dtype=torch.long)
        return tensor_x, tensor_y

train_df = manifest[manifest["partition"] == "train"].reset_index(drop=True)
dev_df = manifest[manifest["partition"] == "dev"].reset_index(drop=True)

train_dataset = RawDataset(train_df, is_train=True)
dev_dataset = RawDataset(dev_df, is_train=False)

train_targets = train_df["is_spoof"].values
class_counts = np.bincount(train_targets)
class_weights = 1.0 / class_counts
sample_weights = torch.FloatTensor(class_weights[train_targets])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train Batches per Epoch: {len(train_loader):,}")
print(f"Dev Batches per Epoch:   {len(dev_loader):,}")


## Step 13: RawNet2-Mini Architecture Implementation with Feature Map Scaling

In [ ]:
class FMSBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1)
        return x * w + x

class RawResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(in_ch)
        self.act1 = nn.LeakyReLU(0.2)
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.act2 = nn.LeakyReLU(0.2)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.pool = nn.MaxPool1d(3)
        self.fms = FMSBlock(out_ch)
        self.downsample = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        res = self.downsample(x)
        out = self.conv1(self.act1(self.bn1(x)))
        out = self.conv2(self.act2(self.bn2(out)))
        out = out + res
        out = self.pool(out)
        return self.fms(out)

class RawNet2Mini(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.sinc_conv = SincConv1D(out_channels=64, kernel_size=129)
        self.pool_init = nn.MaxPool1d(3)
        self.bn_init = nn.BatchNorm1d(64)
        self.act_init = nn.LeakyReLU(0.2)
        self.block1 = RawResBlock(64, 64)
        self.block2 = RawResBlock(64, 128)
        self.block3 = RawResBlock(128, 128)
        self.block4 = RawResBlock(128, 256)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc_latent = nn.Linear(512, 64)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            self.fc_latent,
            nn.LeakyReLU(0.2),
            nn.Linear(64, num_classes)
        )

    def extract_latent(self, x):
        x = torch.abs(self.sinc_conv(x))
        x = self.act_init(self.bn_init(self.pool_init(x)))
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        pooled = torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1).flatten(1)
        return self.fc_latent(pooled)

    def forward(self, x):
        x = torch.abs(self.sinc_conv(x))
        x = self.act_init(self.bn_init(self.pool_init(x)))
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        pooled = torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1)
        return self.classifier(pooled)

model = RawNet2Mini().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")


## Step 14: Forward Pass and Latent Shape Verification

In [ ]:
with torch.no_grad():
    dummy_wav = torch.zeros(2, 1, 64000).to(device)
    dummy_logits = model(dummy_wav)
    dummy_latent = model.extract_latent(dummy_wav)
    print(f"Output Logits Shape: {dummy_logits.shape} (Expected: (2, 2))")
    print(f"Latent Output Shape: {dummy_latent.shape} (Expected: (2, 64))")
    assert dummy_logits.shape == (2, 2)
    assert dummy_latent.shape == (2, 64)


## Step 15: Focal Loss Function and Biometric Metric Formulation

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction="none", label_smoothing=self.label_smoothing)
        p_t = torch.exp(-ce_loss)
        alpha_factor = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        focal_loss = alpha_factor * ((1.0 - p_t) ** self.gamma) * ce_loss
        return focal_loss.mean()

def calculate_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer_val = float((fpr[idx] + fnr[idx]) / 2)
    optimal_thresh = float(np.clip(thresholds[idx], 0.0, 1.0))
    return eer_val, optimal_thresh

def evaluate_network(model, loader, device, use_amp):
    model.eval()
    scores_list, targets_list = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                probs = torch.softmax(model(x_batch), dim=1)[:, 1]
            scores_list.append(probs.cpu().numpy())
            targets_list.append(y_batch.numpy())
    scores = np.concatenate(scores_list)
    targets = np.concatenate(targets_list)
    eer, thresh = calculate_eer(targets, scores)
    auc = float(roc_auc_score(targets, scores))
    return eer, thresh, auc, scores, targets


## Step 16: Model Training and Real-Time Validation Loop (30 Epochs)

In [ ]:
epochs = 30
lr_init = 1e-4
lr_min = 1e-6
weight_decay = 1e-4

criterion = FocalLoss(alpha=0.75, gamma=2.0, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr_init, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=lr_min)
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

save_dir = "/kaggle/working"
os.makedirs(save_dir, exist_ok=True)
best_checkpoint_path = os.path.join(save_dir, "rawnet_raw_best.pth")
history_path = os.path.join(save_dir, "rawnet_raw_history.json")

best_eer = float("inf")
best_auc = 0.0
best_thresh = 0.5
training_history = []

print(f"Commencing RawNet2-Mini Training: {epochs} Epochs | Accelerator: {device} | AMP: {use_amp}")
print("-" * 80)

for epoch in range(1, epochs + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x_batch.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    scheduler.step()

    val_eer, val_thresh, val_auc, _, _ = evaluate_network(model, dev_loader, device, use_amp)
    elapsed = time.time() - start_time

    record = {
        "epoch": epoch,
        "train_loss": round(epoch_train_loss, 5),
        "val_eer": round(val_eer, 5),
        "val_auc": round(val_auc, 5),
        "val_thresh": round(val_thresh, 5),
        "time_sec": round(elapsed, 1)
    }
    training_history.append(record)

    is_best = val_eer < best_eer
    if is_best:
        best_eer = val_eer
        best_auc = val_auc
        best_thresh = val_thresh
        torch.save({
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "eer": best_eer,
            "auc": best_auc,
            "threshold": best_thresh,
            "model_architecture": "RawNet2-Mini"
        }, best_checkpoint_path)

    flag = " [NEW BEST]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{epochs}] | Loss: {epoch_train_loss:.4f} | Dev EER: {val_eer * 100:.2f}% | Dev AUC: {val_auc:.4f} | Time: {elapsed:.0f}s{flag}")

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(training_history, f, indent=2)

print("-" * 80)
print(f"Training Complete. Optimal Dev EER: {best_eer * 100:.2f}% | Dev AUC: {best_auc:.4f} | Threshold: {best_thresh:.4f}")
print(f"Best Model Checkpoint Saved: {best_checkpoint_path}")


## Step 17: Training Loss Trajectory Across 30 Epochs

In [ ]:
ep_list = [h["epoch"] for h in training_history]
losses = [h["train_loss"] for h in training_history]

plt.figure(figsize=(8, 5))
plt.plot(ep_list, losses, color="steelblue", lw=2, marker="o", ms=4, label="Focal Training Loss")
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Loss Value", fontsize=11)
plt.title("RawNet2-Mini Training Loss Trajectory across 30 Epochs", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "07_training_loss_trajectory.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 18: Development Equal Error Rate (EER) Progression Curve

In [ ]:
eers = [h["val_eer"] * 100 for h in training_history]

plt.figure(figsize=(8, 5))
plt.plot(ep_list, eers, color="firebrick", lw=2, marker="s", ms=4, label="Dev EER (%)")
plt.axhline(min(eers), color="gray", linestyle="--", label=f"Lowest EER: {min(eers):.2f}%")
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Equal Error Rate (%)", fontsize=11)
plt.title("RawNet2-Mini Development Equal Error Rate (EER) Progression", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "08_dev_eer_progression_curve.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 19: Receiver Operating Characteristic (ROC) Curve

In [ ]:
checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
_, optimal_thresh, _, final_scores, final_targets = evaluate_network(model, dev_loader, device, use_amp)

fpr, tpr, _ = roc_curve(final_targets, final_scores, pos_label=1)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="darkviolet", lw=2, label=f"RawNet2-Mini (AUC = {checkpoint['auc']:.4f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle=":", label="Random Chance (AUC = 0.5000)")
plt.scatter([checkpoint['eer']], [1 - checkpoint['eer']], color="crimson", s=60, zorder=5, label=f"EER Operating Point ({checkpoint['eer']*100:.2f}%)")
plt.xlabel("False Positive Rate (FPR)", fontsize=11)
plt.ylabel("True Positive Rate (TPR)", fontsize=11)
plt.title("ROC Curve on ASVspoof 2019 Development Partition (RawNet2-Mini)", fontsize=12)
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "09_receiver_operating_characteristic_roc.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 20: Detection Error Tradeoff (DET) Biometric Curve

In [ ]:
fnr = 1.0 - tpr
safe_fpr = np.clip(fpr, 1e-4, 1.0 - 1e-4)
safe_fnr = np.clip(fnr, 1e-4, 1.0 - 1e-4)

plt.figure(figsize=(7, 6))
plt.plot(safe_fpr * 100, safe_fnr * 100, color="darkorange", lw=2, label="RawNet2-Mini DET Profile")
plt.scatter([checkpoint['eer'] * 100], [checkpoint['eer'] * 100], color="crimson", s=60, zorder=5, label=f"EER = {checkpoint['eer']*100:.2f}%")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("False Alarm Rate (%)", fontsize=11)
plt.ylabel("Miss Rate (%)", fontsize=11)
plt.title("Detection Error Tradeoff (DET) Curve (Log Scale)", fontsize=12)
plt.legend(loc="upper right", fontsize=10)
plt.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "10_detection_error_tradeoff_det.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 21: Precision-Recall Curve Analysis

In [ ]:
prec, rec, _ = precision_recall_curve(final_targets, final_scores, pos_label=1)

plt.figure(figsize=(7, 6))
plt.plot(rec, prec, color="teal", lw=2, label="Precision-Recall Trajectory")
plt.xlabel("Recall", fontsize=11)
plt.ylabel("Precision", fontsize=11)
plt.title("Precision-Recall Curve on Development Partition (RawNet2-Mini)", fontsize=12)
plt.legend(loc="lower left", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "11_precision_recall_trajectory.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 22: Confusion Matrix Heatmap at Optimal Operating Threshold

In [ ]:
predicted_binary = (final_scores >= optimal_thresh).astype(int)
cm = confusion_matrix(final_targets, predicted_binary)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title(f"Confusion Matrix (Threshold = {optimal_thresh:.4f})", fontsize=12)
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ["Bonafide (0)", "Spoof (1)"], fontsize=10)
plt.yticks(tick_marks, ["Bonafide (0)", "Spoof (1)"], fontsize=10)

thresh_val = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > thresh_val else "black"
        plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color=color, fontsize=12, fontweight="bold")

plt.ylabel("True Ground Truth", fontsize=11)
plt.xlabel("Predicted Class", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "12_confusion_matrix_optimal_threshold.png"), dpi=300, bbox_inches="tight")
plt.show()

acc = accuracy_score(final_targets, predicted_binary)
p, r, f1, _ = precision_recall_fscore_support(final_targets, predicted_binary, average="binary")
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {p * 100:.2f}%")
print(f"Recall:    {r * 100:.2f}%")
print(f"F1-Score:  {f1:.4f}")


## Step 23: Posterior Score Density Distribution and Class Separation

In [ ]:
bonafide_scores = final_scores[final_targets == 0]
spoof_scores = final_scores[final_targets == 1]

plt.figure(figsize=(9, 5))
plt.hist(bonafide_scores, bins=60, density=True, alpha=0.6, color="steelblue", label=f"Bonafide Scores (N={len(bonafide_scores):,})")
plt.hist(spoof_scores, bins=60, density=True, alpha=0.6, color="firebrick", label=f"Spoof Scores (N={len(spoof_scores):,})")
plt.axvline(optimal_thresh, color="black", linestyle="--", lw=2, label=f"Optimal Decision Boundary ({optimal_thresh:.4f})")

plt.xlabel("Predicted Spoof Posterior Probability", fontsize=11)
plt.ylabel("Probability Density", fontsize=11)
plt.title("Posterior Probability Density on Dev Split (RawNet2-Mini)", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "13_posterior_probability_density.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 24: Granular Attack Taxonomy Vulnerability Analysis (A01 through A06)

In [ ]:
dev_eval_df = dev_df.copy()
dev_eval_df["spoof_score"] = final_scores
dev_eval_df["predicted_label"] = predicted_binary

attack_mapping = {
    "A01": "TTS: Neural Acoustic (AR RNN) + WaveNet",
    "A02": "TTS: Neural Acoustic (AR RNN) + WORLD",
    "A03": "TTS: Concatenative Unit Selection",
    "A04": "VC: Formant / Pitch Shifting + STRAIGHT",
    "A05": "VC: Variational Autoencoder (VAE)",
    "A06": "VC: Transfer Function Regression + WORLD"
}

breakdown_data = []
for atk_id in ["A01", "A02", "A03", "A04", "A05", "A06"]:
    sub = dev_eval_df[dev_eval_df["attack_id"] == atk_id]
    if len(sub) == 0:
        continue
    correct = int((sub["predicted_label"] == 1).sum())
    detection_acc = correct / len(sub)
    mean_prob = float(sub["spoof_score"].mean())
    breakdown_data.append({
        "Attack ID": atk_id,
        "Algorithm Family": attack_mapping.get(atk_id, "Unknown"),
        "Total Utterances": len(sub),
        "Detection Accuracy (%)": round(detection_acc * 100, 2),
        "Mean Spoof Score": round(mean_prob, 4)
    })

bonafide_sub = dev_eval_df[dev_eval_df["key"] == "bonafide"]
bon_correct = int((bonafide_sub["predicted_label"] == 0).sum())
bon_acc = bon_correct / len(bonafide_sub)
breakdown_data.append({
    "Attack ID": "Bonafide",
    "Algorithm Family": "Authentic Human Speech (VCTK)",
    "Total Utterances": len(bonafide_sub),
    "Detection Accuracy (%)": round(bon_acc * 100, 2),
    "Mean Spoof Score": round(float(bonafide_sub["spoof_score"].mean()), 4)
})

breakdown_df = pd.DataFrame(breakdown_data)
print(breakdown_df.to_string(index=False))

plt.figure(figsize=(10, 5))
attack_labels = breakdown_df["Attack ID"].values
accuracies = breakdown_df["Detection Accuracy (%)"].values
bar_colors = ["firebrick" if a.startswith("A") else "steelblue" for a in attack_labels]

bars = plt.bar(attack_labels, accuracies, color=bar_colors, width=0.55)
plt.xlabel("Attack Algorithm Identifier", fontsize=11)
plt.ylabel("Detection Accuracy (%)", fontsize=11)
plt.title("RawNet2-Mini Sensitivity Across ASVspoof 2019 Known Attacks", fontsize=12)
plt.ylim(0, 105)
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, h + 1.5, f"{h:.1f}%", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "14_attack_taxonomy_accuracy_breakdown.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 25: Post-Training Learned SincNet Filterbank Response Adaptation

In [ ]:
learned_filters = model.sinc_conv.get_filters().squeeze().detach().cpu().numpy()
H_learned = np.abs(np.fft.rfft(learned_filters, n=1024, axis=-1))

plt.figure(figsize=(11, 4.5))
for i in range(0, 64, 3):
    plt.plot(freq_axis_sinc, H_learned[i], lw=1.2, alpha=0.75)

plt.xlabel("Frequency (Hz)", fontsize=11)
plt.ylabel("Learned Filter Magnitude", fontsize=11)
plt.title("Post-Training SincNet Filterbank Frequency Responses: Acoustic Resonance Tracking", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "15_sincnet_learned_frequency_responses.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 26: Latent Representation Space Clustering via 2D t-SNE Projection

In [ ]:
sample_indices = []
for atk_id in ["A01", "A02", "A03", "A04", "A05", "A06"]:
    idx_atk = dev_eval_df[dev_eval_df["attack_id"] == atk_id].index.tolist()[:100]
    sample_indices.extend(idx_atk)
bon_idx = dev_eval_df[dev_eval_df["key"] == "bonafide"].index.tolist()[:300]
sample_indices.extend(bon_idx)

sub_df = dev_eval_df.loc[sample_indices].reset_index(drop=True)
subset_ds = RawDataset(sub_df, is_train=False)
subset_loader = DataLoader(subset_ds, batch_size=64, shuffle=False)

embeddings_list = []
model.eval()
with torch.no_grad():
    for xb, _ in subset_loader:
        xb = xb.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            emb = model.extract_latent(xb)
        embeddings_list.append(emb.cpu().numpy())

latent_matrix = np.concatenate(embeddings_list)
print(f"Extracted Latent Embeddings: {latent_matrix.shape}")

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
coords_2d = tsne.fit_transform(latent_matrix)

plt.figure(figsize=(9, 7))
is_bon = sub_df["key"] == "bonafide"
plt.scatter(coords_2d[is_bon, 0], coords_2d[is_bon, 1], color="steelblue", alpha=0.8, s=35, label="Authentic (Bonafide)")

palette = {
    "A01": "firebrick", "A02": "darkorange", "A03": "gold",
    "A04": "forestgreen", "A05": "darkviolet", "A06": "crimson"
}
for atk, col in palette.items():
    mask = sub_df["attack_id"] == atk
    if mask.sum() > 0:
        plt.scatter(coords_2d[mask, 0], coords_2d[mask, 1], color=col, alpha=0.8, s=35, label=f"Spoof {atk}")

plt.xlabel("t-SNE Dimension 1", fontsize=11)
plt.ylabel("t-SNE Dimension 2", fontsize=11)
plt.title("t-SNE 2D Projection of RawNet2-Mini 64-Dimensional Latent Embeddings", fontsize=12)
plt.legend(loc="best", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "16_tsne_latent_embedding_clusters.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 27: Model Explainability through 1D Temporal Waveform Saliency

In [ ]:
demo_spoof_row = dev_df[dev_df["key"] == "spoof"].iloc[0]
raw_demo = read_mono_audio(demo_spoof_row["file_path"])
proc_demo = preprocess_raw_audio(raw_demo, is_train=False)

input_tensor = torch.from_numpy(proc_demo).unsqueeze(0).unsqueeze(0).to(device)
input_tensor.requires_grad = True

model.eval()
model.zero_grad()
pred_logits = model(input_tensor)
spoof_score = pred_logits[0, 1]
spoof_score.backward()

grad = input_tensor.grad.abs().squeeze().detach().cpu().numpy()
kernel_smooth = np.ones(320) / 320.0
saliency_1d = np.convolve(grad, kernel_smooth, mode="same")
saliency_norm = (saliency_1d - saliency_1d.min()) / (saliency_1d.max() - saliency_1d.min() + 1e-8)

time_sec = np.linspace(0, 4.0, len(proc_demo))

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)

axes[0].plot(time_sec, proc_demo, color="gray", lw=0.7)
axes[0].set_title(f"Input Raw Speech Waveform: {demo_spoof_row['audio_id']} (Attack {demo_spoof_row['attack_id']})", fontsize=11)
axes[0].set_ylabel("Amplitude", fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_sec, saliency_norm, color="crimson", lw=1.2)
axes[1].fill_between(time_sec, 0, saliency_norm, color="crimson", alpha=0.25)
axes[1].set_title("1D Temporal Attribution: Deepfake Synthesis Saliency Hotspots", fontsize=11)
axes[1].set_xlabel("Time (seconds)", fontsize=10)
axes[1].set_ylabel("Gradient Saliency", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "17_temporal_saliency_waveform_attribution.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 28: Live End-to-End File Inference Verification

In [ ]:
def predict_audio_file(file_path, target_model, compute_device, threshold):
    target_model.eval()
    raw = read_mono_audio(file_path)
    proc = preprocess_raw_audio(raw, is_train=False)
    tensor = torch.from_numpy(proc).unsqueeze(0).unsqueeze(0).to(compute_device)
    with torch.no_grad():
        with torch.amp.autocast(device_type=compute_device.type, enabled=use_amp):
            prob = torch.softmax(target_model(tensor), dim=1)[0, 1].item()
    decision = "SPOOF (SYNTHETIC)" if prob >= threshold else "BONAFIDE (AUTHENTIC)"
    confidence = prob if prob >= threshold else 1.0 - prob
    return {
        "file_name": os.path.basename(file_path),
        "decision": decision,
        "spoof_probability": round(prob, 5),
        "confidence": f"{confidence * 100:.2f}%"
    }

bon_cand = dev_df[dev_df["key"] == "bonafide"]
spf_cand = dev_df[dev_df["key"] == "spoof"]

bonafide_test_sample = None
for _, r in bon_cand.iterrows():
    if os.path.isfile(r["file_path"]):
        bonafide_test_sample = r["file_path"]
        break
if not bonafide_test_sample:
    bonafide_test_sample = bon_cand.iloc[0]["file_path"]

spoof_test_sample = None
for _, r in spf_cand.iterrows():
    if os.path.isfile(r["file_path"]):
        spoof_test_sample = r["file_path"]
        break
if not spoof_test_sample:
    spoof_test_sample = spf_cand.iloc[0]["file_path"]

print("Case 1: Ground Truth Authentic Speech")
print(json.dumps(predict_audio_file(bonafide_test_sample, model, device, optimal_thresh), indent=2))
print("\nCase 2: Ground Truth Deepfake Speech")
print(json.dumps(predict_audio_file(spoof_test_sample, model, device, optimal_thresh), indent=2))

print("\nGenerated Scientific Artifacts in /kaggle/working:")
print("-" * 65)
for root_path, _, files in os.walk(save_dir):
    for f in sorted(files):
        full_fpath = os.path.join(root_path, f)
        rel_fpath = os.path.relpath(full_fpath, save_dir)
        size_kb = os.path.getsize(full_fpath) / 1024
        print(f"  {rel_fpath:<45} | {size_kb:>9.1f} KB")
